In [1]:
"""
Ejemplo mínimo de uso de GIMLI sobre un DataFrame sintético.

Simula 3 "casos" CFD (AoA distintos) con un Cp distribuido en dos
"familias" de comportamiento (p.ej. zona laminar / zona con shock),
para comprobar que prepare -> select_model -> fit -> predict -> summary
funciona de punta a punta y que los resultados son razonables.
"""

import numpy as np
import pandas as pd

from FotR import GIMLI

rng = np.random.default_rng(0)

rows = []
for aoa in (1.0, 3.0, 5.0):
    n = 200
    # Dos sub-poblaciones de Cp -> deberíamos recuperar 2 clusters por caso
    cp_a = rng.normal(loc=-0.2, scale=0.05, size=n // 2)
    cp_b = rng.normal(loc=-0.8, scale=0.05, size=n // 2)
    cp = np.concatenate([cp_a, cp_b])
    x = rng.uniform(0, 1, size=n)
    rows.append(pd.DataFrame({"AoA": aoa, "x": x, "Cp": cp}))

df = pd.concat(rows, ignore_index=True)
print("df shape:", df.shape)
print(df.head())

df shape: (600, 3)
   AoA         x        Cp
0  1.0  0.195107 -0.193713
1  1.0  0.577688 -0.206605
2  1.0  0.602239 -0.167979
3  1.0  0.962423 -0.194755
4  1.0  0.072265 -0.226783


In [2]:
cluster = GIMLI(
    df,
    variables=["Cp"],
    groups=["AoA"],
    algorithm="gmm",
    cluster_range=range(1, 5),
    covariance_types=("diag",),
    verbose=True,
)

cluster.prepare()
cluster.select_model()
cluster.fit()
cluster.predict()
table = cluster.summary()

print("\nrepr:", cluster)
print("\nsummary table:\n", table)

df_out = cluster.dataset.dataframe
print("\nclustered head:\n", df_out.head())

assert "cluster" in df_out.columns
assert "cluster_proba_max" in df_out.columns
assert (table["recommended_n"] == 2).all(), "Esperaba recuperar 2 clusters por caso"

print("\nOK: ejemplo ejecutado correctamente.")

[GIMLI] prepared 3 group(s), 600 sample(s), 1 feature(s), scaler='standard'.
[GIMLI] group (1.0,): recommended n_components=2, covariance_type='diag' (by bic).
[GIMLI] group (3.0,): recommended n_components=2, covariance_type='diag' (by bic).
[GIMLI] group (5.0,): recommended n_components=2, covariance_type='diag' (by bic).
[GIMLI] fitted 3/3 group(s) using algorithm 'gmm'.
[GIMLI] predicted labels for 600/600 row(s).
 AoA  n_samples  n_clusters_used covariance_type_used  recommended_n recommended_covariance        bic        aic  log_likelihood  silhouette  calinski_harabasz  davies_bouldin      entropy  converged  iterations
 1.0        200                2                 diag              2                   diag 128.537449 112.045862      -51.022931    0.908999        7922.822140        0.128553 2.763102e-11       True           2
 3.0        200                2                 diag              2                   diag 143.112104 126.620517      -58.310259    0.905175        724